# Lecture 11. Graph Neural Networks for Molecules

**PHYG004 · Sogang University · 2026 Spring**
**Instructor:** Prof. Young Woo Choi

## Today's mission

Build a graph neural network from scratch and train it on the same Tetris dataset Friday's lecture (L12) will use. Two failure modes are exposed that motivate equivariance.

1. A flat-coordinate MLP fails on rotated test inputs.
2. A distance-only graph network (SchNet style) is rotation-invariant, but cannot distinguish a chiral pair.

The cliff-hanger: Friday (L12) replaces scalar messages with direction-aware tensorial messages and recovers both.

**New in this version.** After the Tetris toy problem we take SchNet to **real quantum-chemistry data** — a subset of the **QM9** dataset (~134k small organic molecules computed at the DFT B3LYP/6-31G(2df,p) level) — and regress the **atomization energy** $U_0$ from real 3D atomic coordinates and atomic numbers. This is the first time the network meets physical units (eV), and it gives a concrete sense of how far a distance-only model can go on a real benchmark before chirality / directional information becomes the bottleneck.

![water as a graph](images/00_motivation_water_graph.png)

## Roadmap

| Part | Topic |
|---|---|
| 0 | **Geometric DL blueprint**: one symmetry recipe behind CNN / GNN / e3nn |
| 1–2 | install, imports |
| 3–4 | what a graph is; node and edge features |
| 5 | permutation invariance — the constraint that drives architecture choice |
| 6–7 | MPNN three-step recipe (Gilmer et al., 2017). **Checkpoint A** |
| 8 | reproducing the same step with `jraph` |
| 9–10 | Tetris dataset, fully-connected radius graph |
| 11 | a vanilla MLP baseline. Fails on rotation |
| 12 | SchNet — a continuous-filter, distance-only MPNN |
| 13 | training SchNet on Tetris. **Checkpoint B**: 87.5% ceiling |
| 14 | rotation test (passes), chirality test (fails) |
| 15–18 | **QM9 real molecules**: load, train SchNet regressor, **Checkpoint C** |
| 19 | recap → Friday (L12: e3nn on rMD17 real trajectories) |

## Source

* Gilmer et al., 2017. *Neural Message Passing for Quantum Chemistry.* [arXiv:1704.01212](https://arxiv.org/abs/1704.01212).
* Schütt et al., 2017. *SchNet: A continuous-filter convolutional neural network for modeling quantum interactions.* [arXiv:1706.08566](https://arxiv.org/abs/1706.08566).
* Thomas et al., 2018. *Tensor Field Networks.* [arXiv:1802.08219](https://arxiv.org/abs/1802.08219).
* Ramakrishnan et al., 2014. *Quantum chemistry structures and properties of 134 kilo molecules* (the QM9 dataset). [*Sci. Data* **1**, 140022](https://doi.org/10.1038/sdata.2014.22).

## Runtime

The Tetris part uses **Colab CPU** (8 examples; GPU adds latency without speedup). The QM9 part downloads a real dataset (~few hundred MB, cached by `torch_geometric`) and trains a small SchNet for a few minutes on **Colab CPU**; a free **T4 GPU** makes it snappier but is not required for the ~5k-molecule subset we use.


## Part 0. The geometric deep learning blueprint (L11–L13 bridge)

Before any code, one idea ties this week's three lectures together. Every
architecture we will use is the **most general map that respects a symmetry of
the data**. A physicist already knows this move: the *form* of a physical law is
fixed by demanding invariance under a transformation group $G$ (Galilei,
Lorentz, gauge, …). Geometric deep learning is the same demand applied to a
learnable function.

The recipe is always the same three ingredients:

1. a **domain** (grid, set/graph, point cloud in $\mathbb{R}^3$),
2. a **symmetry group** $G$ acting on it,
3. a layer that is **equivariant**: $f(g \cdot x) = \rho(g)\, f(x)$ for all $g \in G$
   (the output transforms the same way the input did), capped by an
   **invariant** readout for graph-level scalars.

| Lecture | Domain | Symmetry group $G$ | Equivariant layer | Invariant readout |
|---|---|---|---|---|
| **L11 (CNN)** | pixel grid | **translations** $\mathbb{Z}^2$ | convolution (weight sharing) | global pooling |
| **L11 (GNN)** | set / graph of atoms | **permutations** $S_N$ | message passing $\oplus$ | sum/mean over nodes |
| **L12 (e3nn)** | point cloud in $\mathbb{R}^3$ | **rotations + reflections** $O(3)$ | tensor-product message | $\ell=0$ (scalar) channels |
| **L13 (MLIP)** | atoms + forces | $E(3)$ = $O(3) \ltimes$ translations | equivariant MPNN | energy (scalar), forces ($\ell=1$) |

Read the table top to bottom and a story emerges. A **CNN** shares one filter
across every position — that *weight sharing* **is** translation equivariance; a
fully-connected net on flat pixels would have to relearn a feature at every
location. A **GNN** replaces the grid by a graph and the translation group by
the permutation group $S_N$: the permutation-invariant aggregator $\oplus$ is
the GNN's analogue of weight sharing. That is today's lecture.

But a GNN that only sees scalar features (atomic numbers, distances) is blind to
**orientation and handedness**. To respect the *full* Euclidean group $E(3)$ —
rotations, reflections, translations — messages must themselves be geometric
tensors that rotate correctly. That is **e3nn** (Friday, L12), and stacking it
into an interatomic potential is **L13**.

> **Where today fits.** This notebook builds the $S_N$ (permutation) rung of the
> ladder and then deliberately walks into the wall at the top of the $O(3)$ rung:
> our distance-only SchNet is rotation-*invariant* but **parity-blind**, so it
> cannot tell a molecule from its mirror image. That failure is the bridge to
> L12.


## Cell 1. Install packages

Six packages.

* `jax`, `jraph`, `flax`, `optax` — core JAX stack and graph utilities.
* `e3nn-jax` — used for one helper today; the equivariant features come on Friday.
* `networkx` — graph visualization.

Do **not** reinstall `jax` itself in Colab; the matched `jaxlib` is already there.


In [ ]:
!uv pip install -q "e3nn-jax>=0.20.8,<0.22" flax optax jraph networkx tqdm matplotlib

## Cell 2. Imports and setup

`float64` keeps numerical sanity checks crisp. A fixed seed makes results reproducible.


In [ ]:
import jax
jax.config.update("jax_enable_x64", True)

import jax.numpy as jnp
import numpy as np
import flax
import flax.linen as nn
import jraph
import optax
import e3nn_jax as e3nn
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import networkx as nx

SEED = 42
key = jax.random.PRNGKey(SEED)

print("jax    :", jax.__version__)
print("jraph  :", jraph.__version__)
print("flax   :", flax.__version__)
print("default dtype:", jnp.zeros(1).dtype)

## Cell 3. The graph data structure

A graph is a pair $G = (V, E)$ with

* a finite set of **nodes** (vertices) $V = \{v_1, v_2, \ldots, v_N\}$,
* a set of **edges** $E \subseteq V \times V$ connecting node pairs.

Each node carries a **feature vector** $h_i \in \mathbb{R}^d$. Each edge $(i, j) \in E$ may carry an **edge feature** $e_{ij}$ (e.g. the interatomic distance).

For a molecule, the natural mapping is the following.

| Graph object | Molecular meaning |
|---|---|
| Node $i$ | atom $i$ |
| Node feature $h_i$ | atomic number, charge, embedding |
| Edge $(i, j)$ | bond, or pair within distance cutoff |
| Edge feature $e_{ij}$ | distance, bond type |

The **neighborhood** of node $i$ is $\mathcal{N}(i) = \{\, j : (i, j) \in E \,\}$.

In code, edges are stored as two parallel arrays of length $|E|$: `senders[k]` and `receivers[k]` give the endpoints of the $k$-th *directed* edge. An undirected edge becomes two directed edges (one in each direction).

![graph anatomy: nodes, edges, features, neighborhood](images/01_graph_anatomy.png)


In [ ]:
# Water (H2O) at experimental geometry, in angstroms.
water_atoms = ["O", "H", "H"]
water_pos = jnp.array([[ 0.00,  0.00, 0.00],
                       [ 0.76,  0.59, 0.00],
                       [-0.76,  0.59, 0.00]])

# Pairwise distance matrix.
dij = jnp.linalg.norm(water_pos[:, None, :] - water_pos[None, :, :], axis=-1)
print("distance matrix:")
print(np.round(np.array(dij), 3))

# Radius graph with cutoff 1.2 A: catches both O-H bonds, excludes the H-H pair (1.52 A).
cutoff = 1.2
mask = (dij < cutoff) & (dij > 0.0)
senders, receivers = jnp.where(mask)
print(f"\nsenders   : {np.array(senders)}")
print(f"receivers : {np.array(receivers)}")
print(f"# directed edges: {senders.shape[0]}  (= 2 bonds * 2 directions)")

# Visualize the graph.
G = nx.Graph()
G.add_nodes_from(range(3))
G.add_edges_from([(int(s), int(r)) for s, r in zip(senders, receivers) if s < r])
fig, ax = plt.subplots(figsize=(4.0, 3.0))
pos2d = {i: (float(water_pos[i, 0]), float(water_pos[i, 1])) for i in range(3)}
species_color = {"O": "#cc4444", "H": "#dddddd"}
nx.draw(G, pos2d,
        node_color=[species_color[a] for a in water_atoms],
        node_size=900, edgecolors="black", linewidths=1.5,
        labels={i: water_atoms[i] for i in range(3)},
        with_labels=True, ax=ax)
ax.set_aspect("equal"); ax.set_title("water as a 3-node, 2-edge graph")
plt.tight_layout(); plt.show()

## Cell 4. Permutation invariance

Atoms have no canonical order. Relabeling them is a bookkeeping change; the molecule is the same. Any model output that represents a *physical property of the graph* must therefore be unchanged under relabeling.

**Definition (graph-level invariance).** Let $\pi$ be a permutation of the $N$ nodes. A function $f$ of node features is *permutation invariant* if
$$f(h_{\pi(1)}, h_{\pi(2)}, \ldots, h_{\pi(N)}) \;=\; f(h_1, h_2, \ldots, h_N) \qquad \forall \pi.$$

**Definition (node-level equivariance).** A function $f$ that produces one output per node is *permutation equivariant* if
$$f\big(h_{\pi(\cdot)}\big)_i \;=\; f(h)_{\pi^{-1}(i)} \qquad \forall \pi.$$

A standard MLP applied to the *flat concatenation* $(h_1, h_2, \ldots, h_N)$ violates both: swapping two atoms produces a different output.

The canonical permutation-invariant operators on a multiset are
$$\bigoplus_{i} h_i \;\in\; \{\,\textstyle\sum_i h_i,\;\; \tfrac{1}{N}\sum_i h_i,\;\; \max_i h_i\,\}.$$

These are the standard **aggregation** operators in graph networks.

| Operator | Property |
|---|---|
| $\sum$ | preserves multiplicity (degree-sensitive) |
| mean | degree-normalized; loses degree information |
| max | selective; ignores all but the largest |

Default below: **sum**.

![permutation invariance: same multiset, two orderings, identical sum](images/02_permutation_invariance.png)


In [ ]:
# Set of 4 features, two orderings.
x = jnp.array([1.0, 2.0, 3.0, 4.0])
perm = jnp.array([2, 0, 3, 1])
x_perm = x[perm]

# Sum / mean / max are permutation invariant.
print(f"sum    : original={x.sum():.3f},  permuted={x_perm.sum():.3f}")
print(f"mean   : original={x.mean():.3f},  permuted={x_perm.mean():.3f}")
print(f"max    : original={x.max():.3f},  permuted={x_perm.max():.3f}")

# Concat-and-Linear (the MLP analogue) is NOT permutation invariant.
W = jax.random.normal(jax.random.PRNGKey(0), (4,))
print(f"\ndot(x,    W) = {jnp.dot(x, W):.4f}")
print(f"dot(x_pi, W) = {jnp.dot(x_perm, W):.4f}   (different)")

## Cell 5. The MPNN recipe (Gilmer et al., 2017)

Almost every modern graph network for molecules — SchNet, DimeNet, PaiNN, NequIP, MACE — follows one template. A single layer is **three steps**.

### Step 1. Message

For each directed edge $(i, j) \in E$, compute
$$m_{ij} \;=\; \phi_M\!\left(h_i,\; h_j,\; e_{ij}\right),$$
with $\phi_M$ a learnable function (typically an MLP). Interpretation: $j$ packages information about itself and the edge $(i,j)$ to send to $i$.

### Step 2. Aggregate

Node $i$ collects all incoming messages with a permutation-invariant operator:
$$M_i \;=\; \bigoplus_{j \in \mathcal{N}(i)} m_{ij}.$$

### Step 3. Update

Combine the previous state with the aggregated message:
$$h_i^{\text{new}} \;=\; \phi_U\!\left(h_i,\; M_i\right),$$
again with a learnable $\phi_U$.

### Stacking layers

Repeating the recipe $L$ times propagates information across $L$-hop neighborhoods. The **receptive field** of node $i$ at layer $L$ is its $L$-hop neighborhood. Choosing $L$ matches the longest correlation length the task demands.

![MPNN three steps: message → aggregate → update](images/03_mpnn_three_steps.png)

![receptive field grows by one hop per layer](images/04_message_passing_propagation.png)


## Cell 6. One MPNN step on water — Checkpoint A

The cell below carries out a single layer by hand on the water graph from Cell 3, then verifies permutation equivariance numerically.

* Node feature: a 4-dim atomic embedding (one-hot-like).
* Edge feature: a single scalar $e_{ij} = \exp(-d_{ij})$.
* Message: $m_{ij} = h_j \cdot e_{ij}$ (a trivial $\phi_M$).
* Aggregation: sum.
* Update: $h_i^{\text{new}} = h_i + M_i$ (residual).


In [ ]:
# Atomic embedding (4-dim, one-hot-like).
embed = {"O": jnp.array([1.0, 0.0, 0.0, 0.0]),
         "H": jnp.array([0.0, 1.0, 0.0, 0.0])}
h = jnp.stack([embed[a] for a in water_atoms])      # shape (3, 4)

# Edge feature: e_ij = exp(-d_ij).
edge_feat = jnp.exp(-dij[senders, receivers])[:, None]   # shape (E, 1)
print(f"h shape   : {h.shape}")
print(f"edge feat : {np.round(np.array(edge_feat.squeeze()), 4)}")

# Step 1. message  m_ij = h_j * e_ij
m_ij = h[senders] * edge_feat                       # (E, 4)

# Step 2. aggregate (sum into receivers)
M_i = jax.ops.segment_sum(m_ij, receivers, num_segments=h.shape[0])

# Step 3. update (residual)
h_new = h + M_i

print(f"\nh_new (after one MPNN step):")
print(np.round(np.array(h_new), 3))

# --- Sanity check: permutation equivariance ---
perm = jnp.array([2, 0, 1])
h_perm = h[perm]
inv_perm = jnp.argsort(perm)
m_p = h_perm[inv_perm[senders]] * edge_feat
M_p = jax.ops.segment_sum(m_p, inv_perm[receivers], num_segments=3)
h_p_new = h_perm + M_p
diff = jnp.max(jnp.abs(h_p_new[inv_perm] - h_new))
print(f"\npermutation-equivariance |diff| = {diff:.2e}")
assert diff < 1e-10
print("\n✅ Checkpoint A: one MPNN step is permutation-equivariant.")

## Cell 7. The same step in `jraph`

`jraph.GraphNetwork` packages the three-step recipe. The user provides:

* `update_edge_fn(edge, sender, receiver, glob)` $\to$ message,
* `update_node_fn(node, sender_msg, receiver_msg, glob)` $\to$ updated node feature,
* `aggregate_edges_for_nodes_fn` (the $\bigoplus$ operator).

`jraph` performs the segment-sum internally and returns a new `GraphsTuple` with updated nodes. The numerical result must match the hand calculation in Cell 6 exactly.


In [ ]:
graph = jraph.GraphsTuple(
    nodes=h, edges=edge_feat,
    senders=senders, receivers=receivers,
    n_node=jnp.array([3]),
    n_edge=jnp.array([senders.shape[0]]),
    globals=None,
)

def edge_fn(edge, sender, receiver, glob):
    return sender * edge       # m_ij = h_j * e_ij

def node_fn(node, sender_msg, receiver_msg, glob):
    return node + receiver_msg # residual update

net = jraph.GraphNetwork(
    update_edge_fn=edge_fn,
    update_node_fn=node_fn,
    update_global_fn=None,
    aggregate_edges_for_nodes_fn=jraph.segment_sum,
)
out = net(graph)
print("jraph-updated nodes:")
print(np.round(np.array(out.nodes), 3))

diff = jnp.max(jnp.abs(out.nodes - h_new))
print(f"\n|jraph - manual| = {diff:.2e}")
assert diff < 1e-10
print("jraph reproduces the manual step exactly.")

## Cell 8. The Tetris dataset

Eight 4-atom shapes on the integer lattice (Thomas et al., 2018, Section 4.1). Friday's lecture uses the same dataset.

| Label | Name | Description |
|---|---|---|
| 0 | `chiral_1` | chiral, one handedness |
| 1 | `chiral_2` | mirror image of `chiral_1` |
| 2 | `square` | 2×2 flat square |
| 3 | `line` | 4 atoms in a row |
| 4 | `corner` | three legs from a corner |
| 5 | `L` | L-shape |
| 6 | `T` | T-shape |
| 7 | `zigzag` | zigzag |

Two facts drive everything below.

**Fact 1 (chiral pair has identical distance multisets).** `chiral_1` and `chiral_2` are mirror images, not rotations of one another. Every pairwise distance is the same:
$$\{r_{ij}\}_{\text{chiral\_1}} = \{r_{ij}\}_{\text{chiral\_2}}.$$
Any model that depends on $\{r_{ij}\}$ alone produces *identical* outputs on both, so the maximum achievable training accuracy is $\tfrac{7}{8} = 87.5\%$.

**Fact 2 (a small radius cutoff hides too much).** With a $1.1$ cutoff (nearest neighbours only), **five** of the eight shapes — `chiral_1`, `chiral_2`, `line`, `L`, and `zigzag` — collapse to the *same* graph-isomorphic 4-vertex **path** ($\,\bullet\!-\!\bullet\!-\!\bullet\!-\!\bullet\,$) with all three edge lengths equal to $1$. A 1-dimensional Weisfeiler–Lehman (WL) test cannot separate isomorphic graphs, so a distance-only MPNN sees these five as one class. The remaining shapes — `square`, `corner`, `T` — give distinct nearest-neighbour graphs. So at $r_\mathrm{cut}=1.1$ the model can resolve only **3 distinguishable classes** (the 5-way collapse counts as one), giving a **maximum achievable training accuracy of $3/8 = 37.5\%$** — *not* $50\%$. We therefore use a **fully-connected graph** (cutoff $4.0$) for the main experiment, so the model receives the entire pairwise-distance multiset and the ceiling rises to the chiral-pair limit $7/8 = 87.5\%$ (Fact 1). Take-home #2 and #3 ask you to verify the $37.5\%$ figure and the 5-way WL collapse in code.

![tetris shapes; chiral_1 and chiral_2 share an identical distance multiset](images/06_tetris_shapes.png)


In [ ]:
shape_names = ["chiral_1", "chiral_2", "square", "line",
               "corner", "L", "T", "zigzag"]
pos = jnp.array([
    [[0, 0, 0], [0, 0, 1], [1, 0, 0], [1, 1, 0]],   # chiral_1
    [[1, 1, 1], [1, 1, 2], [2, 1, 1], [2, 0, 1]],   # chiral_2 = mirror(chiral_1)
    [[0, 0, 0], [1, 0, 0], [0, 1, 0], [1, 1, 0]],   # square
    [[0, 0, 0], [0, 0, 1], [0, 0, 2], [0, 0, 3]],   # line
    [[0, 0, 0], [0, 0, 1], [0, 1, 0], [1, 0, 0]],   # corner
    [[0, 0, 0], [0, 0, 1], [0, 0, 2], [0, 1, 0]],   # L
    [[0, 0, 0], [0, 0, 1], [0, 0, 2], [0, 1, 1]],   # T
    [[0, 0, 0], [1, 0, 0], [1, 1, 0], [2, 1, 0]],   # zigzag
], dtype=jnp.float64)
labels = jnp.arange(8)

# Verify the chiral pair has identical pairwise-distance multisets.
def dist_multiset(p):
    d = jnp.linalg.norm(p[:, None, :] - p[None, :, :], axis=-1)
    iu = jnp.triu_indices(p.shape[0], k=1)
    return jnp.sort(d[iu])

d1, d2 = dist_multiset(pos[0]), dist_multiset(pos[1])
print(f"chiral_1 distances : {np.round(np.array(d1), 3)}")
print(f"chiral_2 distances : {np.round(np.array(d2), 3)}")
print(f"|d1 - d2|_max = {float(jnp.max(jnp.abs(d1 - d2))):.2e}   (= 0)")

# Plot all 8 shapes.
fig = plt.figure(figsize=(14, 7))
for i, name in enumerate(shape_names):
    ax = fig.add_subplot(2, 4, i + 1, projection="3d")
    p = np.asarray(pos[i])
    ax.scatter(p[:, 0], p[:, 1], p[:, 2], s=80)
    for a_idx in range(4):
        for b_idx in range(a_idx + 1, 4):
            if np.linalg.norm(p[a_idx] - p[b_idx]) < 1.1:
                ax.plot(*zip(p[a_idx], p[b_idx]), color="black", lw=1)
    ax.set_title(name)
    ax.set_xlim(-0.5, 2.5); ax.set_ylim(-0.5, 2.5); ax.set_zlim(-0.5, 3.5)
plt.tight_layout(); plt.show()

## Cell 9. Building the batched graph dataset

For each shape we build a fully-connected directed graph (cutoff 4.0 covers the longest distance, $r = 3$ in the line shape). All 8 graphs are stacked into one `jraph.GraphsTuple` for batched processing. Node feature: position; edge feature: built inside the model from positions.


In [ ]:
def radius_graph(p, r_cut):
    n = p.shape[0]
    d = jnp.linalg.norm(p[:, None, :] - p[None, :, :], axis=-1)
    mask = (d < r_cut) & (d > 0.0)
    s, r = jnp.where(mask, size=n * (n - 1), fill_value=-1)
    keep = s >= 0
    return s[keep], r[keep]

def make_tetris_graphs(r_cut=4.0):
    gs = []
    for p, lab in zip(pos, labels):
        s, r = radius_graph(p, r_cut)
        gs.append(jraph.GraphsTuple(
            nodes=p, edges=None, senders=s, receivers=r,
            n_node=jnp.array([4]),
            n_edge=jnp.array([s.shape[0]]),
            globals=lab[None],
        ))
    return jraph.batch(gs)

graphs = make_tetris_graphs(r_cut=4.0)
print(f"total nodes : {int(graphs.n_node.sum())}   (= 8 shapes × 4 atoms)")
print(f"total edges : {int(graphs.n_edge.sum())}   (= 8 shapes × 4 × 3 directed)")
print(f"labels      : {graphs.globals}")

## Cell 10. Baseline 1 — flat-coordinate MLP

The simplest possible baseline. Concatenate the 4 atoms' coordinates into a single 12-dim vector, pass through a 2-hidden-layer MLP. No graph structure, no rotation invariance.

After training to 100% on the original orientations, the **rotation test** evaluates on 100 random rotations of each shape. Expected accuracy: $1/8 = 12.5\%$ — the random-guessing rate. Coordinates rotate, but the MLP treats every rotation as a brand-new input.


In [ ]:
class FlatMLP(nn.Module):
    @nn.compact
    def __call__(self, x):
        x = nn.Dense(128)(x); x = nn.silu(x)
        x = nn.Dense(128)(x); x = nn.silu(x)
        return nn.Dense(8)(x)

X_train = pos.reshape(8, 12)
y_train = labels

mlp = FlatMLP()
mlp_params = mlp.init(jax.random.PRNGKey(0), X_train)
mlp_opt = optax.adam(0.01)
mlp_state = mlp_opt.init(mlp_params)

@jax.jit
def mlp_step(params, state, X, y):
    def loss_fn(p):
        logits = mlp.apply(p, X)
        return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, y)), logits
    grads, logits = jax.grad(loss_fn, has_aux=True)(params)
    updates, state = mlp_opt.update(grads, state)
    return optax.apply_updates(params, updates), state, jnp.mean(jnp.argmax(logits, axis=1) == y)

for s in range(500):
    mlp_params, mlp_state, mlp_train_acc = mlp_step(mlp_params, mlp_state, X_train, y_train)
print(f"MLP train acc            : {float(mlp_train_acc)*100:.1f}%")

# Random rotation matrix (det = +1).
def rand_rotation(key):
    A = jax.random.normal(key, (3, 3))
    Q, _ = jnp.linalg.qr(A)
    return Q * jnp.sign(jnp.linalg.det(Q))

rot_keys = jax.random.split(jax.random.PRNGKey(99), 100)
mlp_rot_acc = []
for k in rot_keys:
    R = rand_rotation(k)
    X_rot = (pos @ R.T).reshape(8, 12)
    pred = jnp.argmax(mlp.apply(mlp_params, X_rot), axis=1)
    mlp_rot_acc.append(float(jnp.mean(pred == labels)))
print(f"MLP rotated-test acc     : {np.mean(mlp_rot_acc)*100:.1f}%   (≈ random 1/8 = 12.5%)")

## Cell 11. Baseline 2 — SchNet (continuous-filter MPNN)

SchNet (Schütt et al., 2017) is the simplest serious MPNN for molecules. Three design choices make it natural and rotation-invariant.

1. **Scalar node features.** Initial $h_i$ is an atomic embedding (here a constant 1, since all atoms are equivalent in Tetris).

2. **Distance-only edge features.** Each edge feature is a smooth radial expansion of the interatomic distance,
$$\mathrm{RBF}(r_{ij}) \;=\; \big[\,e^{-\gamma (r_{ij} - \mu_k)^2}\,\big]_{k=1, \ldots, K}, \qquad \mu_k \in [0, r_{\max}].$$
A small MLP turns this into a learnable filter $W_{ij} = \mathrm{MLP}\big(\mathrm{RBF}(r_{ij})\big)$.

3. **Message by elementwise gating.** The sender feature is multiplied componentwise by the filter,
$$m_{ij} \;=\; \big(W^{(s)} h_j\big) \odot W_{ij}.$$

Because every quantity that enters the network depends only on $r_{ij} = \|\vec r_{ij}\|$, **the entire forward pass is invariant under any rotation of the input coordinates**. Direction information is discarded by construction.

![SchNet message construction: distance → RBF → filter → gate](images/05_schnet_block.png)


## Cell 12. Training SchNet on Tetris — Checkpoint B

We train for 1500 steps. Expected outcomes:

* train accuracy $\to 87.5\%$ exactly,
* `chiral_1` and `chiral_2` produce *identical* logits (their distance multisets are equal — the network cannot tell them apart even in principle),
* the rotated-test accuracy equals the train accuracy (rotation invariance is built in).


In [ ]:
def radius_graph(p, r_cut):
    n = p.shape[0]
    d = jnp.linalg.norm(p[:, None, :] - p[None, :, :], axis=-1)
    mask = (d < r_cut) & (d > 0.0)
    s, r = jnp.where(mask, size=n * (n - 1), fill_value=-1)
    keep = s >= 0
    return s[keep], r[keep]

def make_tetris_graphs(r_cut=4.0):
    gs = []
    for p, lab in zip(pos, labels):
        s, r = radius_graph(p, r_cut)
        gs.append(jraph.GraphsTuple(
            nodes=p, edges=None, senders=s, receivers=r,
            n_node=jnp.array([4]),
            n_edge=jnp.array([s.shape[0]]),
            globals=lab[None],
        ))
    return jraph.batch(gs)

graphs = make_tetris_graphs(r_cut=4.0)
print(f"total nodes : {int(graphs.n_node.sum())}   (= 8 shapes × 4 atoms)")
print(f"total edges : {int(graphs.n_edge.sum())}   (= 8 shapes × 4 × 3 directed)")
print(f"labels      : {graphs.globals}")

In [ ]:
class SchNetLayer(nn.Module):
    hidden_dim: int = 64
    n_rbf: int = 32

    @nn.compact
    def __call__(self, h, gr, positions):
        rij = positions[gr.receivers] - positions[gr.senders]
        dij = jnp.linalg.norm(rij, axis=-1, keepdims=True)
        # radial basis expansion
        mu = jnp.linspace(0.0, 3.5, self.n_rbf)
        gamma = (self.n_rbf / 3.5) ** 2
        rbf = jnp.exp(-gamma * (dij - mu) ** 2)              # (E, n_rbf)
        # filter network
        W = nn.Dense(self.hidden_dim)(rbf); W = nn.silu(W)
        W = nn.Dense(self.hidden_dim)(W)
        # message: linearise sender, gate by filter
        s = nn.Dense(self.hidden_dim)(h[gr.senders])
        m = s * W
        # aggregate
        agg = jraph.segment_sum(m, gr.receivers, h.shape[0])
        # update (residual)
        u = nn.Dense(self.hidden_dim)(agg); u = nn.silu(u)
        u = nn.Dense(self.hidden_dim)(u)
        return h + u


class SchNetTetris(nn.Module):
    hidden_dim: int = 64
    n_layers: int = 4

    @nn.compact
    def __call__(self, gr):
        positions = gr.nodes
        h = jnp.ones((positions.shape[0], 1))
        h = nn.Dense(self.hidden_dim)(h)
        for _ in range(self.n_layers):
            h = SchNetLayer(self.hidden_dim)(h, gr, positions)
        # graph-level pool
        graph_idx = jnp.repeat(jnp.arange(gr.n_node.shape[0]),
                                gr.n_node, total_repeat_length=positions.shape[0])
        graph_h = jraph.segment_sum(h, graph_idx, gr.n_node.shape[0])
        graph_h = nn.Dense(self.hidden_dim)(graph_h); graph_h = nn.silu(graph_h)
        return nn.Dense(8)(graph_h)


schnet = SchNetTetris()
sch_params = schnet.init(jax.random.PRNGKey(SEED), graphs)
sch_opt = optax.adam(0.005)
sch_state = sch_opt.init(sch_params)

@jax.jit
def sch_step(params, state, gr):
    def loss_fn(p):
        logits = schnet.apply(p, gr)
        labs = gr.globals
        return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, labs)), logits
    grads, logits = jax.grad(loss_fn, has_aux=True)(params)
    updates, state = sch_opt.update(grads, state)
    return optax.apply_updates(params, updates), state, logits

@jax.jit
def sch_loss(params, gr):
    logits = schnet.apply(params, gr)
    return jnp.mean(optax.softmax_cross_entropy_with_integer_labels(logits, gr.globals))

print("Training SchNet on Tetris (1500 steps)...")
acc_hist, loss_hist = [], []
for s in tqdm(range(1500)):
    sch_params, sch_state, logits = sch_step(sch_params, sch_state, graphs)
    acc_hist.append(float(jnp.mean(jnp.argmax(logits, axis=1) == labels)))
    loss_hist.append(float(sch_loss(sch_params, graphs)))

preds = jnp.argmax(logits, axis=1)
print(f"\npredictions : {preds}")
print(f"truth       : {labels}")
print(f"train acc   : {acc_hist[-1]*100:.1f}%   (max 87.5%)")

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
axes[0].plot(loss_hist); axes[0].set_xlabel("step"); axes[0].set_ylabel("loss")
axes[1].plot(acc_hist); axes[1].set_xlabel("step"); axes[1].set_ylabel("accuracy")
axes[1].axhline(0.875, color="red", ls="--", lw=1, label="distance-only ceiling 7/8")
axes[1].set_ylim(0, 1.05); axes[1].legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

print("\n✅ Checkpoint B: SchNet hits the distance-only ceiling at 7/8 = 87.5%.")

## Cell 13. Rotation test, chirality inspection

Two probes.

**Rotation test.** Generate 100 random rotation matrices (uniform in $SO(3)$), apply each to the input coordinates, classify. SchNet uses only $r_{ij}$, so its logits are identical before and after rotation. The accuracy on rotated inputs must equal the accuracy on the originals.

**Chirality inspection.** Print the logits for `chiral_1` (label 0) and `chiral_2` (label 1). Their distance multisets are identical, hence the *logits are identical*. The maximum absolute difference is exactly 0.


In [ ]:
# Part 1. SchNet rotation test.
sch_rot_acc = []
for k in rot_keys:
    R = rand_rotation(k)
    pred = jnp.argmax(
        schnet.apply(sch_params, graphs._replace(nodes=graphs.nodes @ R.T)),
        axis=1,
    )
    sch_rot_acc.append(float(jnp.mean(pred == labels)))
print(f"SchNet rotated-test acc  : {np.mean(sch_rot_acc)*100:.1f}%   = train acc, by construction")

# Part 2. Chiral pair logits.
print("\nchiral_1 logits :", np.round(np.array(logits[0]), 3))
print("chiral_2 logits :", np.round(np.array(logits[1]), 3))
diff = float(jnp.max(jnp.abs(logits[0] - logits[1])))
print(f"|chiral_1 - chiral_2|_max = {diff:.3e}   ← exactly 0; the chiral pair is invisible.")

# Summary bar chart.
fig, ax = plt.subplots(figsize=(7, 3.5))
groups = ["MLP\ntrain", "MLP\nrotated", "SchNet\ntrain", "SchNet\nrotated"]
values = [float(mlp_train_acc) * 100, np.mean(mlp_rot_acc) * 100,
          acc_hist[-1] * 100, np.mean(sch_rot_acc) * 100]
colors = ["#888", "#888", "#3366cc", "#3366cc"]
ax.bar(groups, values, color=colors)
ax.axhline(12.5, color="gray", ls=":", lw=1, label="random (1/8)")
ax.axhline(87.5, color="red", ls="--", lw=1, label="distance-only ceiling (7/8)")
ax.set_ylim(0, 105); ax.set_ylabel("accuracy (%)")
ax.legend(loc="lower right", fontsize=8)
plt.tight_layout(); plt.show()

---

## Part 4 — From Tetris to real molecules: QM9

The Tetris experiment was a controlled toy: 8 hand-built shapes, exact ceilings
we could derive on paper. Now we point the *same* SchNet at a real
quantum-chemistry benchmark and ask a quantitative question physicists actually
care about:

> Given a molecule's atomic numbers $Z_i$ and 3D coordinates $\vec r_i$ (a
> relaxed DFT geometry), predict its **atomization energy** $U_0$ in eV.

### The dataset: QM9

**QM9** (Ramakrishnan et al., 2014) is the standard small-molecule benchmark:
**~134k** stable organic molecules with up to 9 heavy atoms (C, N, O, F) plus H.
For each molecule it provides the **DFT-relaxed 3D geometry** and a dozen
properties computed at the **B3LYP/6-31G(2df,p)** level — including the internal
energy $U_0$ at 0 K, the HOMO–LUMO gap, the dipole moment, and more. Licence:
CC BY 4.0.

`torch_geometric.datasets.QM9` downloads and caches it for us, so no manual data
staging is needed.

### Atomization energy, and why we predict it

The raw $U_0$ of a molecule is dominated by a trivial sum of per-atom self-
energies; that offset has nothing to do with bonding and would let a network
"cheat" by just counting atoms. The physically meaningful target is the
**atomization energy**

$$
E_\text{atomization} \;=\; U_0^\text{molecule} \;-\; \sum_{i} E^\text{atom}(Z_i),
$$

the energy released when the isolated atoms assemble into the molecule. We
subtract per-element atomic reference energies (a simple **linear baseline**, the
ML analogue of removing a constant from a Hamiltonian before diagonalising), so
the network only has to learn the *interaction* part. This is standard practice
in machine-learned potentials and dramatically lowers the MAE.

### What success looks like

A small SchNet trained on a **~5k-molecule subset** for a few minutes on Colab
CPU should reach a test **MAE well under 0.5 eV/molecule**. (State-of-the-art
equivariant models on the full 110k train split reach a few **meV** — three
orders of magnitude better — but that needs a GPU and hours; our goal is to see
a real, non-trivial energy surface being fit, and to feel the hyperparameters.)


### Cell 15. Install the QM9 data loader

`torch_geometric` (PyTorch Geometric) ships the QM9 downloader/cacher. We use it
**only to fetch and parse the data** — the model itself stays pure JAX/Flax. On
Colab the first run downloads and processes QM9 (a few minutes, then cached to
disk).


In [ ]:
# Colab: install PyTorch Geometric just for its QM9 dataset loader.
# (torch is preinstalled on Colab; this pulls torch_geometric + rdkit.)
!pip install -q torch_geometric rdkit


### Cell 16. Load and preprocess a QM9 subset

We take a random **5,000-molecule** subset, extract atomic numbers `Z` and 3D
positions `pos` (in angstroms), and read the target `U0` (in eV; PyG already
stores QM9 energies in eV). Then we remove the per-element atomic reference
energy by **ordinary least squares**: fit $U_0 \approx \sum_Z n_Z\, \varepsilon_Z$
on the *train* split (where $n_Z$ is the count of element $Z$ in a molecule), and
predict the residual atomization energy. This linear baseline is the physically
correct "zero" of our energy scale.


In [ ]:
import numpy as np
import jax.numpy as jnp
from torch_geometric.datasets import QM9

# ---- download / load (cached after first run) ----
dataset = QM9(root="/content/qm9")           # ~134k molecules
print(f"full QM9 size: {len(dataset)} molecules")

# QM9 target index 7 is U0 (internal energy at 0 K), already in eV in PyG.
U0_INDEX = 7
N_SUBSET = 5000
N_TRAIN  = 4000                              # remaining 1000 -> test

rng = np.random.default_rng(SEED)
idx = rng.permutation(len(dataset))[:N_SUBSET]

# Allowed elements in QM9: H, C, N, O, F.
Z_LIST = [1, 6, 7, 8, 9]
Z_TO_COL = {z: c for c, z in enumerate(Z_LIST)}
MAX_ATOMS = 29                               # QM9 max (with H)

mols = []      # list of (Z array, pos array, U0 scalar)
for j in idx:
    d = dataset[int(j)]
    Z   = d.z.numpy().astype(np.int32)       # (n_atoms,)
    pos = d.pos.numpy().astype(np.float64)   # (n_atoms, 3) angstrom
    U0  = float(d.y[0, U0_INDEX])            # eV
    mols.append((Z, pos, U0))

# ---- linear atomic-reference baseline (fit on train only) ----
def composition_vector(Z):
    v = np.zeros(len(Z_LIST))
    for z in Z:
        v[Z_TO_COL[int(z)]] += 1.0
    return v

C_train = np.stack([composition_vector(mols[i][0]) for i in range(N_TRAIN)])
y_train_raw = np.array([mols[i][2] for i in range(N_TRAIN)])
# eps[Z] solving  C @ eps ~ U0  (least squares).
eps, *_ = np.linalg.lstsq(C_train, y_train_raw, rcond=None)
print("fitted atomic reference energies (eV) per element:")
for z, e in zip(Z_LIST, eps):
    print(f"  Z={z:2d}: {e:10.4f} eV")

def atomization_energy(Z, U0):
    return U0 - composition_vector(Z) @ eps

# Targets become small residual atomization energies (eV).
targets = np.array([atomization_energy(Z, U0) for (Z, _, U0) in mols])
print(f"\natomization-energy target: mean={targets.mean():.3f} eV, "
      f"std={targets.std():.3f} eV")
print(f"  (raw U0 std was {np.array([m[2] for m in mols]).std():.1f} eV — "
      f"the baseline removed the huge per-atom offset)")

# Standardize the regression target for stable training; undo at eval.
TARGET_MEAN = targets[:N_TRAIN].mean()
TARGET_STD  = targets[:N_TRAIN].std()
print(f"\nshapes — example molecule 0: Z {mols[0][0].shape}, pos {mols[0][1].shape}")


### Cell 17. Batch QM9 molecules into a single `jraph.GraphsTuple`

Each molecule becomes a fully-connected radius graph (cutoff $5.0$ Å — large
enough to span any QM9 molecule's bonded neighbourhood while keeping edge counts
manageable). Unlike Tetris, the **node feature is now the atomic number** (via a
learnable embedding indexed by `Z`), because real molecules contain several
elements. We bucket molecules into fixed-size **mini-batches** with
`jraph.batch`, which concatenates node/edge arrays and offsets the sender/
receiver indices automatically.


In [ ]:
import jraph

R_CUT_QM9 = 5.0   # angstrom

def mol_to_graph(Z, pos, target_std):
    # Build a fully-connected radius graph for one molecule.
    #   nodes : (n_atoms,) int32 column index into the Z-embedding table.
    #   target: standardized atomization energy stored in `globals`.
    p = jnp.asarray(pos)
    d = jnp.linalg.norm(p[:, None, :] - p[None, :, :], axis=-1)
    mask = (d < R_CUT_QM9) & (d > 0.0)
    n = p.shape[0]
    s, r = jnp.where(mask, size=n * (n - 1), fill_value=-1)
    keep = s >= 0
    s, r = s[keep], r[keep]
    z_col = jnp.array([Z_TO_COL[int(z)] for z in Z], dtype=jnp.int32)
    return jraph.GraphsTuple(
        nodes=z_col, edges=None, senders=s, receivers=r,
        n_node=jnp.array([n]), n_edge=jnp.array([s.shape[0]]),
        globals=jnp.array([[target_std]]),          # (1, 1)
    ), p

# Precompute per-molecule graphs and positions (positions stored separately,
# because SchNet needs the raw coordinates to build distances).
def standardize(t):
    return (t - TARGET_MEAN) / TARGET_STD

graphs_all, pos_all = [], []
for (Z, pos, U0), t in zip(mols, targets):
    g, p = mol_to_graph(Z, pos, standardize(t))
    graphs_all.append(g)
    pos_all.append(p)

train_g = graphs_all[:N_TRAIN]
train_p = pos_all[:N_TRAIN]
test_g  = graphs_all[N_TRAIN:]
test_p  = pos_all[N_TRAIN:]
print(f"train molecules: {len(train_g)},  test molecules: {len(test_g)}")

def batch_molecules(gs, ps):
    # jraph.batch the graphs AND concatenate positions in the same order.
    batched = jraph.batch(gs)
    positions = jnp.concatenate(ps, axis=0)        # (sum n_atoms, 3)
    return batched, positions

# Sanity check on a tiny batch.
b_g, b_p = batch_molecules(train_g[:4], train_p[:4])
print(f"batch of 4 -> total atoms {int(b_g.n_node.sum())}, "
      f"total edges {int(b_g.n_edge.sum())}, positions {b_p.shape}")
print(f"node feature (Z column indices) sample: {np.array(b_g.nodes[:8])}")


### Cell 18. SchNet **regressor** for QM9

Almost identical to the Tetris classifier, with three changes for real
molecules:

1. **Atomic embedding.** `nn.Embed(num_embeddings=5, features=hidden_dim)` maps
   each element (H, C, N, O, F) to a learnable vector — the node feature is now
   chemistry-aware, not a constant `1`.
2. **Per-atom energy readout, then sum.** Total energy is *extensive*: it scales
   with the number of atoms. So the head outputs a **scalar per atom** and we
   `segment_sum` over each molecule. This is the standard "the energy is a sum of
   atomic contributions" decomposition used in every ML potential (L13).
3. **Scalar regression head** (1 output) instead of 8-way logits.

The RBF cutoff is widened to match `R_CUT_QM9`. Everything else — distance →
RBF → filter → gate → residual update — is the SchNet block you already trained
on Tetris, so the architecture transfers directly.


In [ ]:
import flax.linen as nn
import optax
import jax

class QM9SchNetLayer(nn.Module):
    hidden_dim: int = 64
    n_rbf: int = 32
    r_cut: float = R_CUT_QM9

    @nn.compact
    def __call__(self, h, gr, positions):
        rij = positions[gr.receivers] - positions[gr.senders]
        dij = jnp.linalg.norm(rij, axis=-1, keepdims=True)
        # radial basis expansion out to the cutoff
        mu = jnp.linspace(0.0, self.r_cut, self.n_rbf)
        gamma = (self.n_rbf / self.r_cut) ** 2
        rbf = jnp.exp(-gamma * (dij - mu) ** 2)            # (E, n_rbf)
        # cosine envelope -> smooth decay to 0 at the cutoff
        env = 0.5 * (jnp.cos(jnp.pi * dij / self.r_cut) + 1.0) * (dij < self.r_cut)
        rbf = rbf * env
        # learnable filter
        W = nn.Dense(self.hidden_dim)(rbf); W = nn.silu(W)
        W = nn.Dense(self.hidden_dim)(W)
        # message: linearise sender, gate by filter
        s = nn.Dense(self.hidden_dim)(h[gr.senders])
        m = s * W
        agg = jraph.segment_sum(m, gr.receivers, h.shape[0])
        u = nn.Dense(self.hidden_dim)(agg); u = nn.silu(u)
        u = nn.Dense(self.hidden_dim)(u)
        return h + u                                       # residual update


class QM9SchNet(nn.Module):
    hidden_dim: int = 64
    n_layers: int = 3
    n_rbf: int = 32

    @nn.compact
    def __call__(self, gr, positions):
        # node feature: learnable atomic embedding indexed by element column.
        h = nn.Embed(num_embeddings=len(Z_LIST),
                     features=self.hidden_dim)(gr.nodes)    # (n_atoms, hidden)
        for _ in range(self.n_layers):
            h = QM9SchNetLayer(self.hidden_dim, self.n_rbf)(h, gr, positions)
        # per-atom scalar energy contribution
        atom_e = nn.silu(nn.Dense(self.hidden_dim)(h))
        atom_e = nn.Dense(1)(atom_e)                        # (n_atoms, 1)
        # extensive readout: sum atomic energies within each molecule
        graph_idx = jnp.repeat(jnp.arange(gr.n_node.shape[0]),
                               gr.n_node, total_repeat_length=positions.shape[0])
        e = jraph.segment_sum(atom_e, graph_idx, gr.n_node.shape[0])  # (B, 1)
        return e.squeeze(-1)                                # (B,)


print("QM9SchNet defined:",
      "hidden_dim=64, n_layers=3, n_rbf=32, per-atom extensive readout.")


### Cell 19. Train and evaluate — **Checkpoint C**

We train on the 4,000-molecule train split with mini-batch Adam for a few
epochs and report the **test MAE in eV/molecule** (de-standardized back to
physical units). The success criterion is **MAE < 0.5 eV/molecule**.

> **Hyperparameter sweep (the Checkpoint-C exercise).** The three knobs
> `hidden_dim`, `n_layers`, `n_rbf` are exposed at the top of the cell. Run the
> default once, record the test MAE, then change one knob at a time and see what
> moves. Bigger/deeper is usually better but slower; `n_rbf` controls how finely
> the model resolves distances. Keep a little table of (config → test MAE).


In [ ]:
import time

# ---- Checkpoint-C knobs: change these for the sweep ----
HIDDEN_DIM = 64
N_LAYERS   = 3
N_RBF      = 32
BATCH_SIZE = 32
N_EPOCHS   = 12
LR         = 1e-3
# --------------------------------------------------------

model = QM9SchNet(hidden_dim=HIDDEN_DIM, n_layers=N_LAYERS, n_rbf=N_RBF)

# init on one batch
init_g, init_p = batch_molecules(train_g[:BATCH_SIZE], train_p[:BATCH_SIZE])
params = model.init(jax.random.PRNGKey(SEED), init_g, init_p)
n_params = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"model parameters: {n_params:,}")

opt = optax.adam(LR)
opt_state = opt.init(params)

@jax.jit
def train_step(params, opt_state, gr, positions, y):
    def loss_fn(p):
        pred = model.apply(p, gr, positions)
        return jnp.mean((pred - y) ** 2)
    loss, grads = jax.value_and_grad(loss_fn)(params)
    updates, opt_state = opt.update(grads, opt_state)
    return optax.apply_updates(params, updates), opt_state, loss

@jax.jit
def predict(params, gr, positions):
    return model.apply(params, gr, positions)

def iterate_batches(gs, ps, bs, shuffle_key=None):
    n = len(gs)
    order = np.arange(n)
    if shuffle_key is not None:
        order = np.array(jax.random.permutation(shuffle_key, n))
    for k in range(0, n - bs + 1, bs):
        sel = order[k:k + bs]
        yield batch_molecules([gs[i] for i in sel], [ps[i] for i in sel])

def evaluate_mae_eV(params):
    # Test MAE in physical eV (undo standardization).
    errs = []
    for gr, positions in iterate_batches(test_g, test_p, BATCH_SIZE):
        pred = predict(params, gr, positions)            # standardized
        y    = gr.globals.squeeze(-1)
        pred_eV = pred * TARGET_STD + TARGET_MEAN
        y_eV    = y    * TARGET_STD + TARGET_MEAN
        errs.append(np.abs(np.array(pred_eV - y_eV)))
    return float(np.concatenate(errs).mean())

print(f"\nTraining QM9 SchNet: {N_EPOCHS} epochs, batch {BATCH_SIZE}, lr {LR}")
key_e = jax.random.PRNGKey(SEED)
for epoch in range(N_EPOCHS):
    key_e, sk = jax.random.split(key_e)
    t0 = time.time()
    losses = []
    for gr, positions in iterate_batches(train_g, train_p, BATCH_SIZE, sk):
        y = gr.globals.squeeze(-1)
        params, opt_state, loss = train_step(params, opt_state, gr, positions, y)
        losses.append(float(loss))
    mae = evaluate_mae_eV(params)
    print(f"epoch {epoch+1:2d} | train MSE {np.mean(losses):.4f} "
          f"| test MAE {mae:.3f} eV | {time.time()-t0:.1f}s")

final_mae = evaluate_mae_eV(params)
print(f"\nfinal test MAE = {final_mae:.3f} eV/molecule")
assert final_mae < 0.5, "Checkpoint C target is MAE < 0.5 eV — train longer or grow the model."
print("✅ Checkpoint C: QM9 atomization-energy MAE < 0.5 eV/molecule.")


In [ ]:
# Parity plot: predicted vs DFT atomization energy on the test set.
import matplotlib.pyplot as plt

preds_eV, true_eV = [], []
for gr, positions in iterate_batches(test_g, test_p, BATCH_SIZE):
    p = np.array(predict(params, gr, positions)) * TARGET_STD + TARGET_MEAN
    t = np.array(gr.globals.squeeze(-1)) * TARGET_STD + TARGET_MEAN
    preds_eV.append(p); true_eV.append(t)
preds_eV = np.concatenate(preds_eV)
true_eV  = np.concatenate(true_eV)

fig, ax = plt.subplots(figsize=(4.5, 4.5))
lo, hi = true_eV.min(), true_eV.max()
ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="ideal")
ax.scatter(true_eV, preds_eV, s=8, alpha=0.4)
ax.set_xlabel("DFT atomization energy (eV)")
ax.set_ylabel("SchNet prediction (eV)")
ax.set_title(f"QM9 test set — MAE {final_mae:.3f} eV")
ax.set_aspect("equal"); ax.legend(loc="upper left", fontsize=8)
plt.tight_layout(); plt.show()


### What QM9 just taught us

* The **same** distance-only SchNet that maxed out at 87.5% on a toy now fits a
  real DFT energy surface to a fraction of an eV — energies that took hours of
  quantum chemistry per molecule are reproduced in microseconds.
* Removing the **atomic-reference baseline** (a physics prior, not an ML trick)
  was what made the regression easy: the network only models *interactions*, the
  same way you subtract a known constant from a Hamiltonian before solving it.
* Atomization energy is **extensive**, so the readout is a **sum of per-atom
  contributions** — the architectural seed of every machine-learned interatomic
  potential in L13.
* And yet: SchNet here is still **rotation-invariant but parity-blind**. On QM9
  that is mostly harmless (few exact mirror-image pairs with different
  $U_0$ at this level of theory), but for **forces**, **dipoles**, and **chiral
  energetics** you need messages that carry *direction*. That is precisely what
  Friday's e3nn lecture (L12) builds — on real rMD17 molecular trajectories.


## Cell 14. Recap and the cliff-hanger

| Test | MLP | SchNet | What it shows |
|---|---|---|---|
| Train accuracy | 100% | 87.5% | SchNet bottlenecked by the chiral pair |
| Rotated test | $\approx 12\%$ | 87.5% | rotation invariance only when *built into the architecture* |
| Chiral pair | (random) | **identical logits** | distance-only is parity-blind |

Two failure modes, two architectural causes.

* The MLP fails on rotation because flat coordinates are not a rotation-invariant representation, and 8 examples are too few to learn rotation by augmentation.
* SchNet fails on chirality because the distance multiset $\{r_{ij}\}$ is *parity-symmetric*: it does not change under reflection.

The cure is to let messages carry **direction**, not just distance, and to track parity explicitly. That is the program of equivariant message passing — **Friday's lecture (L12)**, where the scalar message $m_{ij} = (W^{(s)}h_j)\odot W_{ij}$ is replaced by a *tensorial* message built from spherical harmonics $Y^{(\ell)}(\hat r_{ij})$ and Clebsch–Gordan tensor products, so that $\ell=1$ (vector) and pseudo-scalar channels finally distinguish the chiral pair. **L12 applies exactly this machinery to real molecular trajectories from the rMD17 dataset** (MD17 recomputed at higher accuracy), and **L13** stacks it into a full machine-learned interatomic potential. Today's cliff-hanger — the identical chiral logits you just printed — is resolved on Friday.

![cliffhanger: Friday adds direction-aware messages](images/09_cliffhanger.png)

## Optional take-home exercises

**1. Aggregation operator (chiral collapse).** Replace `segment_sum` by
`segment_mean` (i.e. `jraph.segment_mean`) in the SchNet layer. Predict first:
does the chiral pair still collapse to identical logits? Then verify in code.
*(Hint: chirality survives in neither operator — both are symmetric functions of
the same distance multiset.)*

**2. Cutoff sensitivity → 37.5%.** Re-run the SchNet training with
`make_tetris_graphs(r_cut=1.1)`. Using the graph-isomorphism argument in Fact 2,
predict the maximum train accuracy ($3/8 = 37.5\%$, the **corrected** ceiling —
not 50%), then check that training plateaus there.

**3. Weisfeiler–Lehman collapse.** Implement the 1-dimensional WL colour-
refinement test on the nearest-neighbour ($r_\mathrm{cut}=1.1$) graphs of all 8
shapes and confirm that **five** of them — `chiral_1`, `chiral_2`, `line`, `L`,
`zigzag` — share the same final WL colouring (the 4-vertex path), while
`square`, `corner`, `T` are distinct. This is the structural reason behind the
37.5% ceiling of exercise 2.

**4. Rotation augmentation.** Train the flat MLP with on-the-fly random rotations
applied to each batch. How many augmented copies per shape are needed to reach
$> 90\%$ on a held-out rotated test? Compare the cost to SchNet's *zero*
augmentation (rotation invariance is free, by construction).

**5. QM9 hyperparameter sweep (Checkpoint C).** In the QM9 section below, vary
`hidden_dim`, `n_layers`, and `n_rbf` and record the test MAE. Which knob buys
the most accuracy per parameter?

## References

* Gilmer, Schoenholz, Riley, Vinyals, Dahl. *Neural Message Passing for Quantum Chemistry.* ICML 2017.
* Schütt, Kindermans, Sauceda, Chmiela, Tkatchenko, Müller. *SchNet: A continuous-filter convolutional neural network for modeling quantum interactions.* NeurIPS 2017. [arXiv:1706.08566](https://arxiv.org/abs/1706.08566). *(This is the SchNet model used above.)*
* Schütt, Arbabzadah, Chmiela, Müller, Tkatchenko. *Quantum-chemical insights from deep tensor neural networks (DTNN).* [*Nat. Commun.* **8**, 13890 (2017)](https://doi.org/10.1038/ncomms13890). *(The earlier DTNN model — a distinct architecture, listed separately to avoid the common mis-citation.)*
* Ramakrishnan, Dral, Rupp, von Lilienfeld. *Quantum chemistry structures and properties of 134 kilo molecules (QM9).* [*Sci. Data* **1**, 140022 (2014)](https://doi.org/10.1038/sdata.2014.22).
* Thomas, Smidt, Kearnes, Yang, Li, Kohlhoff, Riley. *Tensor Field Networks.* arXiv:1802.08219, 2018.
* Xu, Hu, Leskovec, Jegelka. *How Powerful are Graph Neural Networks?* ICLR 2019.
